# 06 — Compare recent and baseline productivity state

Measure whether mean NDVI during the recent part of a reporting window differs from its earlier baseline.


## Inputs, outputs, and execution order

**Inputs:** Annual rasters from 04; default windows contain 13 baseline years followed by 3 recent years.

**Outputs:** State Z, five-class, three-class, and binary rasters named `ndvi_state_*_endYYYY.tif` under `state/`.

Run cells from top to bottom in a fresh kernel. Open this notebook beside `workflow_config.json` and `workflow_support.py`. The configuration uses Mion study settings; the original boundary and final criteria table were not supplied. See the README for the small example and for adapting the workflow.

### How to interpret this step

The implemented statistic is Z = (recent mean − baseline mean) / (baseline standard deviation / √recent_len). The baseline moves with each 16-year window. Minimum valid counts are 10/13 and 2/3 by default; the denominator retains √3 when only two recent values are valid. Constant baselines and invalid statistics become nodata. Binary degradation uses Z < −1.96; the three-class product uses ±1.28.


## 1. Import packages

Use the Digital Earth Africa Sandbox for steps that access the data cube. Local raster processing uses the packages listed in `environment.yml`.


In [ ]:
from __future__ import annotations

# =========================================================
# IMPORTS
# =========================================================
import os
import time
import re
import gc
import glob
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Iterable

import numpy as np
import rasterio

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


## 2. Load shared settings

The settings below describe the run. Edit the JSON configuration before continuing; later parameter cells explain the settings used by this notebook.


In [ ]:
# Edit workflow_config.json once, then restart and rerun each notebook.
from workflow_support import load_config, require_files, load_aoi, assert_same_grid
cfg = load_config()
print(f"Study: {cfg['study_id']} | Years: {cfg['start_year']}-{cfg['end_year']}")
print(f"Output directory: {cfg['output_dir']}")


## 3. Set baseline, recent period, and completeness

Keep these periods consistent with trajectory. Tighten minimum valid counts to require complete data if the study design needs it. DDOF=0 retains the population-standard-deviation convention of the original.


In [ ]:
# =========================================================
# USER-DEFINED PARAMETERS
# =========================================================
OUTPUT = cfg["output_dir"]
ANNUAL_PATTERN = f"{OUTPUT}/annual_ndvi/ndvi_annual_{cfg['annual_metric']}_*.tif"
OUTPUT_DIR = f"{OUTPUT}/state"
OUTPUT_PREFIX = "ndvi_state"

BASELINE_LEN = cfg["state_baseline_len"]
RECENT_LEN = cfg["state_recent_len"]
STEP_YEARS = cfg["report_step_years"]                 # 4 for reporting cycle, 1 for yearly
END_YEARS = None               # e.g. [2015, 2019, 2023] or None for automatic

MIN_BASELINE_VALID = cfg["state_min_baseline_valid"]
MIN_RECENT_VALID = cfg["state_min_recent_valid"]
DDOF = cfg["state_ddof"]

BINARY_DEGRADED_THRESHOLD = cfg["state_binary_threshold"]
SKIP_EXISTING = cfg["skip_existing"]


## 4. Read annual inputs

State stacks a complete reporting period in RAM. Alignment includes CRS and pixel transform, not only raster shape.


In [ ]:
# =========================================================
# HELPER: MEMORY CLEANUP
# =========================================================
def clear_memory(*names):
    gc.collect()


# =========================================================
# DATA CLASS
# =========================================================
@dataclass
class RasterMeta:
    profile: dict
    nodata: Optional[float]
    height: int
    width: int


# =========================================================
# REGEX
# =========================================================
_YEAR_RE = re.compile(r"(19|20)\d{2}")


# =========================================================
# FILE DISCOVERY
# =========================================================
def discover_year_paths(pattern: str) -> Dict[int, str]:
    """
    Find annual raster files and return:
        {year: filepath}
    """
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files matched pattern: {pattern}")

    year_to_path: Dict[int, str] = {}

    for fp in files:
        fname = os.path.basename(fp)
        m = _YEAR_RE.search(fname)
        if not m:
            raise ValueError(f"Could not parse year from filename: {fname}")

        year = int(m.group(0))

        if year in year_to_path:
            raise ValueError(
                f"Duplicate year {year} found:\n"
                f"  {year_to_path[year]}\n"
                f"  {fp}"
            )

        year_to_path[year] = fp

    return year_to_path


# =========================================================
# RASTER READ / WRITE HELPERS
# =========================================================
def read_raster_as_float32(path: str) -> Tuple[np.ndarray, RasterMeta]:
    """
    Read a single-band raster as float32 and convert nodata to NaN.
    """
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        nodata = src.nodata
        profile = src.profile.copy()
        height, width = src.height, src.width

    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)

    arr = np.where(np.isfinite(arr), arr, np.nan)
    meta = RasterMeta(profile=profile, nodata=nodata, height=height, width=width)

    return arr, meta


def load_year_stack(year_to_path: Dict[int, str], years: List[int]) -> Tuple[np.ndarray, RasterMeta]:
    """
    Load a stack of annual rasters for selected years.
    Returns:
        stack: (T, H, W)
        meta: RasterMeta
    """
    assert_same_grid([year_to_path[y] for y in years])
    arrays: List[np.ndarray] = []
    ref_meta: Optional[RasterMeta] = None

    for year in years:
        if year not in year_to_path:
            raise KeyError(f"Missing raster for year {year}")

        arr, meta = read_raster_as_float32(year_to_path[year])

        if ref_meta is None:
            ref_meta = meta
        else:
            if (meta.height != ref_meta.height) or (meta.width != ref_meta.width):
                raise ValueError(
                    f"Shape mismatch for year {year}: "
                    f"{(meta.height, meta.width)} != {(ref_meta.height, ref_meta.width)}"
                )

        arrays.append(arr)

    stack = np.stack(arrays, axis=0)
    return stack, ref_meta


def write_geotiff(path: str, array2d: np.ndarray, base_profile: dict, nodata_value=None, dtype=None):
    """
    Write a 2D array to GeoTIFF using base raster georeferencing.
    """
    out_dir = os.path.dirname(path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)

    profile = base_profile.copy()

    if dtype is None:
        dtype = array2d.dtype

    dtype_str = np.dtype(dtype).name
    height, width = array2d.shape

    profile.update(
        driver="GTiff",
        count=1,
        height=height,
        width=width,
        dtype=dtype_str,
        compress="deflate",
        tiled=True,
        blockxsize=512,
        blockysize=512,
        BIGTIFF="IF_SAFER",
    )

    if nodata_value is None:
        nodata_value = -9999.0 if np.issubdtype(np.dtype(dtype), np.floating) else 255

    profile.update(nodata=nodata_value)

    out = array2d.copy()

    if np.issubdtype(np.dtype(dtype), np.floating):
        out = out.astype(np.float32)
        out = np.where(np.isfinite(out), out, nodata_value).astype(np.float32)
    else:
        out = np.where(np.isfinite(out), out, nodata_value).astype(np.dtype(dtype))

    with rasterio.open(path, "w", **profile) as dst:
        dst.write(out, 1)


## 5. Calculate state Z and class maps

Missing annual pixel values are excluded from means and standard deviations, subject to the minimum valid counts. Equality at a threshold follows the explicit comparison operators below.


In [ ]:
# =========================================================
# PRODUCTIVITY STATE CALCULATION
# =========================================================
def productivity_state_z(
    annual_stack: np.ndarray,
    baseline_len: int,
    recent_len: int,
    min_baseline_valid: int,
    min_recent_valid: int,
    ddof: int = 0,
) -> np.ndarray:
    """
    Compute productivity state Z-score:
        Z = (mean(recent) - mean(baseline)) / (std(baseline) / sqrt(recent_len))
    """
    total_len, _, _ = annual_stack.shape
    expected_len = baseline_len + recent_len

    if total_len != expected_len:
        raise ValueError(f"Expected stack length {expected_len}, got {total_len}")

    baseline = annual_stack[:baseline_len]
    recent = annual_stack[baseline_len:]

    n_base = np.sum(np.isfinite(baseline), axis=0)
    n_recent = np.sum(np.isfinite(recent), axis=0)

    baseline_mean = np.nanmean(baseline, axis=0)
    baseline_std = np.nanstd(baseline, axis=0, ddof=ddof)

    recent_mean = np.nanmean(recent, axis=0)

    standard_error = baseline_std / np.sqrt(recent_len)
    z = (recent_mean - baseline_mean) / standard_error

    invalid = (
        (n_base < min_baseline_valid)
        | (n_recent < min_recent_valid)
        | (~np.isfinite(z))
        | (standard_error == 0)
    )

    z = np.where(invalid, np.nan, z).astype(np.float32)
    return z


# =========================================================
# CLASSIFICATION FUNCTIONS
# =========================================================
def classify_state_5(z: np.ndarray) -> np.ndarray:
    """
    5-class productivity state.
    """
    out = np.full(z.shape, 255, dtype=np.uint8)
    valid = np.isfinite(z)
    zz = z[valid]

    cls = np.full(zz.shape, 3, dtype=np.uint8)
    cls[zz < -1.96] = 1
    cls[(zz >= -1.96) & (zz < -1.28)] = 2
    cls[(zz > 1.28) & (zz <= 1.96)] = 4
    cls[zz > 1.96] = 5

    out[valid] = cls
    return out


def classify_state_3(z: np.ndarray) -> np.ndarray:
    """
    3-class productivity state.
    """
    out = np.full(z.shape, 255, dtype=np.uint8)
    valid = np.isfinite(z)
    zz = z[valid]

    cls = np.full(zz.shape, 2, dtype=np.uint8)
    cls[zz < -1.28] = 1
    cls[zz > 1.28] = 3

    out[valid] = cls
    return out


def classify_state_binary_degraded(z: np.ndarray, threshold: float = -1.96) -> np.ndarray:
    """
    Binary degraded map.
      1 = degraded
      0 = not degraded
      255 = nodata
    """
    out = np.full(z.shape, 255, dtype=np.uint8)
    valid = np.isfinite(z)
    out[valid] = (z[valid] < threshold).astype(np.uint8)
    return out


## 6. Define reporting windows

Each end year requires baseline_len + recent_len consecutive files. With the default years and four-year step, the end years are 2015, 2019, and 2023.


In [ ]:
# =========================================================
# REPORTING PERIOD HELPERS
# =========================================================
def required_years_for_end_year(end_year: int, baseline_len: int, recent_len: int) -> List[int]:
    """
    Get required years for a reporting end year.
    """
    total = baseline_len + recent_len
    start_year = end_year - (total - 1)
    return list(range(start_year, end_year + 1))


def feasible_end_years(
    available_years: Iterable[int],
    baseline_len: int,
    recent_len: int,
    step_years: int,
) -> List[int]:
    """
    Return all valid end years for which all required rasters exist.
    """
    years = sorted(set(available_years))
    if not years:
        return []

    min_year = min(years)
    max_year = max(years)

    total = baseline_len + recent_len
    earliest_end_year = min_year + (total - 1)

    candidates = list(range(earliest_end_year, max_year + 1, step_years))
    available_set = set(years)

    valid_end_years = []
    for end_year in candidates:
        req = required_years_for_end_year(end_year, baseline_len, recent_len)
        if all(y in available_set for y in req):
            valid_end_years.append(end_year)

    return valid_end_years


## 7. Write each reporting period

Floating Z rasters use −9999 nodata; categorical state maps use 255.


In [ ]:
# =========================================================
# MAIN PROCESSING FOR ONE END YEAR
# =========================================================
def compute_productivity_state_for_end_year(
    year_to_path: Dict[int, str],
    end_year: int,
    out_dir: str,
    prefix: str,
    baseline_len: int,
    recent_len: int,
    min_baseline_valid: int,
    min_recent_valid: int,
    ddof: int,
    binary_threshold: float,
    skip_existing: bool = True,
):
    """
    Compute productivity state outputs for one end year.
    """
    years = required_years_for_end_year(end_year, baseline_len, recent_len)

    z_fp = os.path.join(out_dir, f"{prefix}_z_end{end_year}.tif")
    c5_fp = os.path.join(out_dir, f"{prefix}_class5_end{end_year}.tif")
    c3_fp = os.path.join(out_dir, f"{prefix}_class3_end{end_year}.tif")
    b_fp = os.path.join(out_dir, f"{prefix}_binary_end{end_year}.tif")

    if skip_existing and all(os.path.exists(fp) for fp in [z_fp, c5_fp, c3_fp, b_fp]):
        print(f"Skipping end year {end_year}: outputs already exist")
        return {
            "end_year": end_year,
            "z": z_fp,
            "class5": c5_fp,
            "class3": c3_fp,
            "binary": b_fp,
        }

    stack = None
    z = None
    cls5 = None
    cls3 = None
    binary_map = None
    meta = None

    try:
        stack, meta = load_year_stack(year_to_path, years)

        z = productivity_state_z(
            annual_stack=stack,
            baseline_len=baseline_len,
            recent_len=recent_len,
            min_baseline_valid=min_baseline_valid,
            min_recent_valid=min_recent_valid,
            ddof=ddof,
        )

        cls5 = classify_state_5(z)
        cls3 = classify_state_3(z)
        binary_map = classify_state_binary_degraded(z, threshold=binary_threshold)

        os.makedirs(out_dir, exist_ok=True)

        write_geotiff(z_fp, z, meta.profile, nodata_value=-9999.0, dtype=np.float32)
        write_geotiff(c5_fp, cls5, meta.profile, nodata_value=255, dtype=np.uint8)
        write_geotiff(c3_fp, cls3, meta.profile, nodata_value=255, dtype=np.uint8)
        write_geotiff(b_fp, binary_map, meta.profile, nodata_value=255, dtype=np.uint8)

        return {
            "end_year": end_year,
            "z": z_fp,
            "class5": c5_fp,
            "class3": c3_fp,
            "binary": b_fp,
        }

    finally:
        stack = None
        z = None
        cls5 = None
        cls3 = None
        binary_map = None
        meta = None
        clear_memory()


## 8. Assemble the reporting series

An empty set of feasible periods raises an error; it does not produce a valid assessment.


In [ ]:
# =========================================================
# MAIN REPORTING SERIES
# =========================================================
def compute_productivity_state_reporting_series(
    annual_pattern: str,
    out_dir: str,
    prefix: str,
    baseline_len: int,
    recent_len: int,
    step_years: int,
    end_years: Optional[List[int]] = None,
    min_baseline_valid: int = 10,
    min_recent_valid: int = 2,
    ddof: int = 0,
    binary_threshold: float = -1.96,
    skip_existing: bool = True,
):
    """
    Run productivity state for all valid reporting periods.
    """
    year_to_path = discover_year_paths(annual_pattern)
    available_years = sorted(year_to_path.keys())

    if end_years is None:
        end_years = feasible_end_years(
            available_years=available_years,
            baseline_len=baseline_len,
            recent_len=recent_len,
            step_years=step_years,
        )

    if not end_years:
        raise ValueError(
            "No feasible end years found. "
            f"Available years: {available_years}"
        )

    results = []

    for end_year in end_years:
        print("=" * 60)
        print(f"Processing end year: {end_year}")
        print("=" * 60)

        result = compute_productivity_state_for_end_year(
            year_to_path=year_to_path,
            end_year=end_year,
            out_dir=out_dir,
            prefix=prefix,
            baseline_len=baseline_len,
            recent_len=recent_len,
            min_baseline_valid=min_baseline_valid,
            min_recent_valid=min_recent_valid,
            ddof=ddof,
            binary_threshold=binary_threshold,
            skip_existing=skip_existing,
        )

        results.append(result)

        # explicit cleanup after each loop
        result = None
        clear_memory()

    return results


## 9. Run the state assessment

Use the binary output matching the selected report end year in notebook 08. Keep the five-class output for interpretation and sensitivity analysis.


In [ ]:
# =========================================================
# RUN
# =========================================================
results = compute_productivity_state_reporting_series(
    annual_pattern=ANNUAL_PATTERN,
    out_dir=OUTPUT_DIR,
    prefix=OUTPUT_PREFIX,
    baseline_len=BASELINE_LEN,
    recent_len=RECENT_LEN,
    step_years=STEP_YEARS,
    end_years=END_YEARS,
    min_baseline_valid=MIN_BASELINE_VALID,
    min_recent_valid=MIN_RECENT_VALID,
    ddof=DDOF,
    binary_threshold=BINARY_DEGRADED_THRESHOLD,
    skip_existing=SKIP_EXISTING,
)

print("\nCompleted outputs:")
for item in results:
    print(item)


## Acknowledgements

We gratefully acknowledge the [Group on Earth Observations Land Degradation Neutrality (GEO-LDN) initiative](https://geo-ldn.org/) for its support of this research and the development of this land productivity monitoring workflow.

We acknowledge **Digital Earth Africa (DE Africa)** for providing the open satellite data, analytical tools, and Sandbox platform used in this work.

We also acknowledge the [University of Energy and Natural Resources (UENR), Ghana](https://uenr.edu.gh/), for its support of this research.


## Check the outputs before continuing

- Review warnings and errors, and confirm that every requested year or reporting period was written.
- Open an output raster and inspect its CRS, extent, nodata mask, and value range.
- Keep the configuration and software versions with your results. Run `python scripts/capture_environment.py` from the repository root.
- If you change the AOI, dates, metric, or thresholds, use a fresh output directory to avoid reusing older results.

See [README.md](README.md), [data requirements](docs/DATA_REQUIREMENTS.md), and [method details](docs/METHODS.md).

The supplied notebooks identify their code as Apache License 2.0; retain the original Digital Earth Africa attribution when reusing it. Dataset licences and acknowledgements apply separately. Use the Issues tab of the repository hosting this workflow for workflow-specific questions.

**Validation status:** documentation, syntax, and synthetic-data checks are described in `docs/VALIDATION.md`. This revised workflow has not been executed against the live Digital Earth Africa data cube. A displayed current date or package version is not evidence of a successful full run.
